In [ ]:
# HyperActivation desktop UI refactor

This notebook is now a thin entry-point that exercises the standalone Tkinter UI located in `hyper_activation/`. The heavy lifting (data access, metrics, statistics, and rendering) has been broken into importable modules so the interface can run outside of Jupyter and without any HTML widgets.

In [ ]:
from hyper_activation.data_access import DataRepository
from hyper_activation.measurements import MeasurementService
from hyper_activation.models import SelectionState

In [ ]:
repo = DataRepository()
df = repo.subject_df
hippo_volume = repo.hippo_volumes
print(f"Loaded {len(df)} subjects and {len(hippo_volume)} hippocampal records.")

In [ ]:
ms = MeasurementService(repo)
state = SelectionState()
sample_activation = ms.series_for_modality("RSP_Activation", state)
sample_activation.head()

In [ ]:
orientation = ms.series_for_modality("OrientationPerformance", state)
connectivity = ms.series_for_modality("Connectivity", state)
print(orientation.describe())
print(connectivity.describe())

In [ ]:
state = SelectionState(conn_units="Bilaterals", conn_stat="count", conn_threshold=0.32)
conn_bilat = ms.series_for_modality("Connectivity", state)
conn_bilat.head()

In [ ]:
hippo_mean = ms.series_for_modality("HippocampiVolume", SelectionState())
hippo_mean.describe()

In [ ]:
> Task connectivity remains a placeholder in the new modular design. See `hyper_activation/measurements.py` for the explicit `NotImplementedError` and add the computation there when the definition is available.

In [ ]:
## Module overview

- `hyper_activation.data_access` loads pickled project resources with a cached repository.
- `hyper_activation.measurements` exposes the measurement service used by both the UI and this notebook.
- `hyper_activation.stats_utils` keeps statistical helpers together.
- `hyper_activation.ui_app` provides the Tkinter window and `launch_app()` helper.
- `run_app.py` is a simple CLI entry-point you can execute outside Jupyter.

HTML(value='\n<style>\n.widget-label, .widget-select select, .widget-select-multiple select, \n.widget-button,…

In [ ]:
from hyper_activation import launch_app

# Running the next line opens the standalone Tkinter window.
# Uncomment to start the interactive UI when you are ready.
# launch_app()


In [ ]:
### Extending measurements

All axis data now flows through a single `MeasurementService.series_for_modality(modality, SelectionState)` call, so adding an axis only requires:

1. Implementing the underlying series generator inside `hyper_activation.measurements`.
2. Updating `AXES_NAMES` and the `series_for_modality` dispatcher.
3. Optionally wiring new controls into `HyperActivationApp`.

In [ ]:
### Data alignment

The helper logic that used to live in `readBasicData` is now encapsulated by `HyperActivationApp._read_basic_data`, ensuring the same behavior whether you trigger the UI or call the service functions directly from Python.

In [ ]:
### Statistics module

`hyper_activation.stats_utils` centralizes the probability helpers. Both `calc_p_classification` and the task-connectivity routine carry explicit placeholder comments so that future contributors know exactly where to plug in improved methodology.

In [ ]:
### Color mapping

`make_color` still linearly maps `z` values, but now falls back to a neutral palette whenever `z` is constant, avoiding the division-by-zero that previously happened inside the notebook.

In [ ]:
### Event loop

The Tkinter button callbacks mirror the old `on_go_click` behavior, but now surface exceptions via a modal dialog so they are visible even when the UI runs outside of a notebook context.


In [ ]:
> Use `python run_app.py` (or call `launch_app()` from this notebook) to open the independent window.

In [ ]:
---
Notebook last updated to document the modular desktop UI refactor on {{DATE}}.
